### EXP 1

In [21]:
import gensim.downloader as api
model=api.load("glove-wiki-gigaword-50")

In [23]:
print(model.most_similar(positive=['walking','swim'],negative=['walk']))

[('swimming', 0.8071774244308472), ('swimmers', 0.7504438757896423), ('swims', 0.7454535365104675), ('surfing', 0.7409682273864746), ('biking', 0.7364492416381836), ('swam', 0.731138288974762), ('jumping', 0.7218191027641296), ('diving', 0.7187682390213013), ('kayaking', 0.7121444344520569), ('snowboarding', 0.7015390992164612)]


In [24]:
result=model.most_similar(positive=['king','woman'],negative=['man'])
print(f"'king'+'man'-'woman' is closest to:{result[0]}")

'king'+'man'-'woman' is closest to:('queen', 0.8523604273796082)


### EXP 2

In [1]:
import gensim.downloader as api
import numpy as np
from numpy.linalg import norm
print("Loading pre-trained word vectors...")
word_vectors=api.load("word2vec-google-news-300")
def explore_word_relationships(word1,word2,word3):
    try:
        vec1=word_vectors[word1]
        vec2=word_vectors[word2]
        vec3=word_vectors[word3]
        result_vector=vec1-vec2+vec3
        similar_words=word_vectors.similar_by_vector(result_vector,topn=10)
        filtered_words=[(word,similarity) for word,similarity in similar_words if word not in {word1,word2,word3}]
        print(f"\nWord Relationship :{word1}-{word2}+{word3}")
        print("Most similar words to the result(excluding input words):")
        for word,similarity in filtered_words[:5]:
            print(f"{word}:{similarity:4f}")
    except KeyError as e:
        print(f"Error:{e} not found in the vocabulary.")
explore_word_relationships("king","man","women")        

Loading pre-trained word vectors...

Word Relationship :king-man+women
Most similar words to the result(excluding input words):
queen:0.535494
kings:0.516231
queens:0.499536
kumaris:0.492385
princes:0.462333


### EXP 2

In [15]:
import gensim.downloader as api
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
import numpy as np

In [17]:
model=api.load("glove-wiki-gigaword-50")

In [26]:

words = ['queen','throne','prince','daughter','elizabeth','princess','kingdom','monarch','eldest','widow']
vector = np.array([model[word]for word in words])
fig = make_subplots(rows = 1,cols=2,subplot_titles = ('PCA Visualization','t-SNE Visualization'),horizontal_spacing = 0.1)
pca=PCA(n_components=2)
pca_vector = pca.fit_transform(vector)
tsne = TSNE(n_components=2,perplexity= 5, n_iter = 250 , random_state = 42)
tsne_vector = tsne.fit_transform(vector)
fig.add_trace(go.Scatter(x = pca_vector[:,0],y = pca_vector[:,1],mode = 'markers+text',text = words, textposition = "top center",marker = dict(size = 12,color = 'purple',symbol = 'diamond',line = dict(color='gold',width=2)),name = 'Words'),row=1,col=1)
fig.add_trace(go.Scatter(x = tsne_vector[:,0],y = tsne_vector[:,1],mode = 'markers+text',text = words, textposition = "top center",marker = dict(size = 12,color = 'purple',symbol = 'diamond',line = dict(color='gold',width=2)),name = 'Words'),row=1,col=2)
fig.update_layout(height = 600, width =1200,showlegend  = False , template = 'plotly_white',title_text = "Royal/Monarchy Word Embeddings Visualization",title_x = 0.5,font = dict(size=12),plot_bgcolor = 'rgba(240,240,255,0.95)',)
fig.update_xaxes(showgrid=True,gridwidth = 1,gridcolor = 'lightgrey')
fig.update_yaxes(showgrid=True,gridwidth = 1,gridcolor = 'lightgrey')
fig.show()              

Exception ignored on calling ctypes callback function: <function _ThreadpoolInfo._find_modules_with_dl_iterate_phdr.<locals>.match_module_callback at 0x7f0ee2c47d00>
Traceback (most recent call last):
  File "/home/ai/anaconda3/lib/python3.10/site-packages/threadpoolctl.py", line 400, in match_module_callback
    self._make_module_from_path(filepath)
  File "/home/ai/anaconda3/lib/python3.10/site-packages/threadpoolctl.py", line 515, in _make_module_from_path
    module = module_class(filepath, prefix, user_api, internal_api)
  File "/home/ai/anaconda3/lib/python3.10/site-packages/threadpoolctl.py", line 606, in __init__
    self.version = self.get_version()
  File "/home/ai/anaconda3/lib/python3.10/site-packages/threadpoolctl.py", line 646, in get_version
    config = get_config().split()
AttributeError: 'NoneType' object has no attribute 'split'


### EXP 3

In [1]:
from gensim.models import Word2Vec
from gensim.utils import simple_preprocess
sentences=["This is a legal document about contracts.",
           "The court will review the legal case.",
           "Medical professionals required specific training.",
           "This is a medical report about the patient."]
tokenized_sentences=[simple_preprocess(sentence)for sentence in sentences]
model=Word2Vec(sentences=tokenized_sentences,vector_size=50,window=5,min_count=1,workers=4,epochs=10)
print(model.wv.most_similar("legal"))

[('case', 0.19041602313518524), ('document', 0.045007091015577316), ('contracts', -0.010094511322677135), ('the', -0.014259982854127884), ('report', -0.02316688746213913), ('court', -0.043792981654405594), ('will', -0.044073157012462616), ('review', -0.09419942647218704), ('patient', -0.12276068329811096), ('required', -0.14990590512752533)]


### EXP 4

In [9]:
!pip install transformers

import gensim.downloader as api
from transformers import pipeline
import nltk
import string
from nltk.tokenize import word_tokenize

nltk.download('punkt')

print("Loading pre-trained word vectors...")
word_vectors = api.load("glove-wiki-gigaword-100")


def replace_keyword_in_prompt(prompt, keyword, word_vectors, topn=1):
    words = word_tokenize(prompt)
    enriched_words = []

    for word in words:
        cleaned_word = word.lower().strip(string.punctuation)

        if cleaned_word == keyword.lower():
            try:
                similar_words = word_vectors.most_similar(cleaned_word, topn=topn)

                if similar_words:
                    replacement_word = similar_words[0][0]
                    print(f"Replacing '{word}' -> '{replacement_word}'")
                    enriched_words.append(replacement_word)
                    continue

            except KeyError:
                print(f"Word '{cleaned_word}' not found in vocabulary. Using original word.")

        enriched_words.append(word)

    enriched_prompt = " ".join(enriched_words)
    print(f"\nEnriched prompt: {enriched_prompt}")

    return enriched_prompt


print("\nLoading GPT-2 model...")
generator = pipeline("text-generation", model="gpt2")


def generate_response(prompt, max_length=100):
    try:
        response = generator(prompt, max_length=max_length, num_return_sequences=1)
        return response[0]['generated_text']

    except Exception as e:
        print(f"Error generating response: {e}")
        return None
original_prompt = "who is king"
print(f"\nOriginal prompt: {original_prompt}")

key_term = "king"

enriched_prompt = replace_keyword_in_prompt(original_prompt, key_term, word_vectors)

print("\nGenerating response for the original prompt...")
original_response = generate_response(original_prompt)

print("\nOriginal prompt response:")
print(original_response)

print("\nGenerating response for the enriched prompt...")
enriched_response = generate_response(enriched_prompt)

print("\nEnriched prompt response:")
print(enriched_response)

print("\nComparison of responses")

print("\nOriginal prompt response length:", len(original_response))
print("Enriched prompt response length:", len(enriched_response))

print("\nOriginal prompt response detail:", original_response.count("."))
print("Enriched prompt response detail:", enriched_response.count("."))

[nltk_data] Downloading package punkt to /home/ai/nltk_data...
[nltk_data]   Package punkt is already up-to-date!


Loading pre-trained word vectors...

Loading GPT-2 model...


Downloading:   0%|          | 0.00/665 [00:00<?, ?B/s]

Downloading:   0%|          | 0.00/548M [00:00<?, ?B/s]

Downloading:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

...

Downloading: 0.00B [00:00, ?B/s]

Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.



Original prompt: who is king
Replacing 'king' -> 'prince'

Enriched prompt: who is prince

Generating response for the original prompt...


Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.



Original prompt response:
who is king]

So, I shall ask a king, I shall ask what are the fruits of which you are an example, that I may bear with him, that all may remember me. And this I believe because my heart desires the fulfillment I cannot desire, because it is the fulfillment of my Father's faith.

4:4 And if I do not see my Father in any of the things which I have seen, yet do not believe, because I am lost in them

Generating response for the enriched prompt...

Enriched prompt response:
who is prince of the throne). They used their magic skill to break through the ice and make a pact with the demons to end all the trials and keep the evil from getting out; it wasn't quite as if they were the Chosen among them but when one of them dies they must leave, and they don't know who to go pick the last one away. I guess it's one hell of a series that seems to have a lot to do with "Maiden of the Shadowlands". If

Comparison of responses

Original prompt response length: 400
Enriched

### EXP 5

In [ ]:
import gensim.downloader as api
import random
import nltk
nltk.download('punkt') 

def generate_paragraph(seed_word):
    similar_words = get_similar_words(seed_word, top_n=5)
    if not similar_words:
        return "Could not generate a paragraph. Try another seed word."
    paragraph = [generate_sentence(seed_word, similar_words) for _ in range(6)]
    return " ".join(paragraph)
seed_word = input("Enter a seed word: ")
paragraph = generate_paragraph(seed_word)
print("\nGenerated Paragraph:\n")
print(paragraph)

In [3]:
import gensim.downloader as api
import random
import nltk 
nltk.download('punkt')

[nltk_data] Downloading package punkt to /home/ai/nltk_data...
[nltk_data]   Package punkt is already up-to-date!


True

In [ ]:
word=api.load("glove-wiki-gigaword-100")
print("word vectors loaded successfully")
word_vectors = api.load("glove-wiki-gigaword-100") 
print("Word vectors loaded successfully!")
def get_similar_words(seed_word, top_n=5):
    try:
        similar_words = word_vectors.most_similar(seed_word, topn=top_n)
        return [word[0] for word in similar_words]
    except KeyError:
        print(f"'{seed_word}' not found in vocabulary. Try another word.")
        return []
def generate_sentence(seed_word, similar_words):
    sentence_templates = [
    f"The {seed_word} was surrounded by {similar_words[0]} and {similar_words[1]}.",
    f"People often associate {seed_word} with {similar_words[2]} and {similar_words[3]}.",
    f"In the land of {seed_word}, {similar_words[4]} was a common sight.",
    f"A story about {seed_word} would be incomplete without {similar_words[1]} and {similar_words[3]}.",
 ]
    return random.choice(sentence_templates)